# Data cleaning and processing

**Objectif du notebook:**
Ce notebook est la suite du notebook *1.Data visualization and analysis*. Il vise à nettoyer les jeux de données reçus, normaliser certains paramètres, créer certaines variables (features engineering) et certaines autres étapes de pré-processing

---

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import json
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack, csr_matrix
from scipy import sparse

## 1. Chargement des données

In [2]:
df_train = pd.read_csv("../data/train.csv")
df_test = pd.read_csv("../data/test.csv")

train = df_train.copy()
test = df_test.copy()

## 2. Data cleaning

### 2.1 Valeurs NaN

Comme vu précédemment, les valeurs manquantes initiales sont principalement au niveau
de la colonne "MaritalStatus". D'autres valeurs manquantes peuvent être générées
volontairement lors du nettoyage des anomalies.

In [3]:
train["MaritalStatus"] = train["MaritalStatus"].fillna("U")
test["MaritalStatus"] = test["MaritalStatus"].fillna("U")

### 2.2 Dates

In [4]:
date_cols = ["DateTimeOfAccident", "DateReported"]

for col in date_cols:
    train[col] = pd.to_datetime(train[col])
    test[col] = pd.to_datetime(test[col])

### 2.3 Suppression des anomalies

Petite précision, on ne supprime pas les outliers, mais des valeurs qui sont erronées car n'ayant pas de sens.

#### 2.3.1 Semaines de plus de 168h

In [5]:
for df in [train, test]:

    df.loc[
        df["HoursWorkedPerWeek"] > 168,
        "HoursWorkedPerWeek"
    ] = np.nan

#### 2.3.2 Jours de plus de 24h

In [6]:
train_hours_per_day = (
    train["HoursWorkedPerWeek"] /
    train["DaysWorkedPerWeek"].replace(0, np.nan)
)

test_hours_per_day = (
    test["HoursWorkedPerWeek"] /
    test["DaysWorkedPerWeek"].replace(0, np.nan)
)

train.loc[
    train_hours_per_day > 24,
    "HoursWorkedPerWeek"
] = np.nan

test.loc[
    test_hours_per_day > 24,
    "HoursWorkedPerWeek"
] = np.nan

#### 2.3.3 Rapport avant accident

In [7]:
train.loc[
    train["DateReported"] < train["DateTimeOfAccident"],
    "DateReported"
] = pd.NaT

test.loc[
    test["DateReported"] < test["DateTimeOfAccident"],
    "DateReported"
] = pd.NaT

In [8]:
print(train.shape)
print(test.shape)

(54000, 15)
(36000, 14)


### 2.4 Feature engineering

#### 2.4.1 Délai entre l'accident et le rapport

In [9]:
for df in [train, test]:

    df["ReportingDelay"] = (
        df["DateReported"] -
        df["DateTimeOfAccident"]
    ).dt.days

median_delay = train["ReportingDelay"].median()

train["ReportingDelay"] = (
    train["ReportingDelay"].fillna(median_delay)
)

test["ReportingDelay"] = (
    test["ReportingDelay"].fillna(median_delay)
)

#### 2.4.2 Heures par jour

In [10]:
train["HoursPerDay"] = (
    train["HoursWorkedPerWeek"] /
    train["DaysWorkedPerWeek"].replace(0, np.nan)
)

test["HoursPerDay"] = (
    test["HoursWorkedPerWeek"] /
    test["DaysWorkedPerWeek"].replace(0, np.nan)
)

#### 2.4.2 Salaire horaire

In [11]:
train["HourlyWage"] = (
    train["WeeklyWages"] /
    train["HoursWorkedPerWeek"].replace(0, np.nan)
)

test["HourlyWage"] = (
    test["WeeklyWages"] /
    test["HoursWorkedPerWeek"].replace(0, np.nan)
)

#### 2.4.3 Date

In [12]:
for df in [train, test]:

    df["AccidentYear"] = df["DateTimeOfAccident"].dt.year
    df["AccidentMonth"] = df["DateTimeOfAccident"].dt.month
    df["AccidentDay"] = df["DateTimeOfAccident"].dt.day
    df["AccidentWeekday"] = df["DateTimeOfAccident"].dt.weekday
    df["AccidentHour"] = df["DateTimeOfAccident"].dt.hour

In [13]:
num_cols = train.select_dtypes(include=["number"]).columns.drop(
    "UltimateIncurredClaimCost"
)

medians = train[num_cols].median()

train[num_cols] = train[num_cols].fillna(medians)
test[num_cols] = test[num_cols].fillna(medians)


cat_cols = [
    "Gender",
    "MaritalStatus",
    "PartTimeFullTime"
]

modes = train[cat_cols].mode().iloc[0]

train[cat_cols] = train[cat_cols].fillna(modes)
test[cat_cols] = test[cat_cols].fillna(modes)

## 2.5 NLP

### 2.5.1 Nettoyage du texte

In [14]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

for df in [train, test]:
    df["ClaimDescription_clean"] = (
        df["ClaimDescription"].apply(clean_text)
    )

### 2.5.2 TF-IDF

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=2000,
    ngram_range=(1,2),
    min_df=5
)

X_train_text = tfidf.fit_transform(
    train["ClaimDescription_clean"]
)

X_test_text = tfidf.transform(
    test["ClaimDescription_clean"]
)

## 2.6 Encoding

In [16]:
label_encoders = {}

for col in cat_cols:

    le = LabelEncoder()

    le.fit(train[col].astype(str))

    train[col] = le.transform(
        train[col].astype(str)
    )

    test[col] = test[col].map(
        lambda x: (
            le.transform([str(x)])[0]
            if str(x) in le.classes_
            else -1
        )
    )

    label_encoders[col] = le

In [17]:
cols_to_drop = [
    "ClaimNumber"
]

X_train_tab = train.drop(
    columns=cols_to_drop + ["UltimateIncurredClaimCost"],
    errors="ignore"
).select_dtypes(include=["number"])

X_test_tab = test.drop(
    columns=cols_to_drop,
    errors="ignore"
).select_dtypes(include=["number"])

# =========================
# Conversion sparse
# =========================

X_train_tab_sparse = csr_matrix(
    X_train_tab.values
)

X_test_tab_sparse = csr_matrix(
    X_test_tab.values
)

# =========================
# Fusion tabulaire + TF-IDF
# =========================

X_train = hstack([
    X_train_tab_sparse,
    X_train_text
])

X_test = hstack([
    X_test_tab_sparse,
    X_test_text
])

# =========================
# Target
# =========================

y_train = train["UltimateIncurredClaimCost"]

# Vérification
print("X_train :", X_train.shape)
print("X_test :", X_test.shape)
print("y_train :", y_train.shape)

X_train : (54000, 2018)
X_test : (36000, 2018)
y_train : (54000,)


# 3. Rapide nettoyage des colonnes et exportation

In [18]:
pd.set_option("display.max_columns", None)

In [19]:
train.head(3)

,ClaimNumber,DateTimeOfAccident,DateReported,Age,Gender,MaritalStatus,DependentChildren,DependentsOther,WeeklyWages,PartTimeFullTime,HoursWorkedPerWeek,DaysWorkedPerWeek,ClaimDescription,InitialIncurredCalimsCost,UltimateIncurredClaimCost,ReportingDelay,HoursPerDay,HourlyWage,AccidentYear,AccidentMonth,AccidentDay,AccidentWeekday,AccidentHour,ClaimDescription_clean
0,WC8285054,2002-04-09 07:00:00+00:00,2002-07-05 00:00:00+00:00,48,1,0,0,0,500.00,0,38.0,5,LIFTING TYRE INJURY TO RIGHT ARM AND WRIST INJURY,1500,4748.203388,86.0,7.6,13.157895,2002,4,9,1,7,lifting tyre injury to right arm and wrist injury
1,WC6982224,1999-01-07 11:00:00+00:00,1999-01-20 00:00:00+00:00,43,0,0,0,0,509.34,0,37.5,5,STEPPED AROUND CRATES AND TRUCK TRAY FRACTURE ...,5500,6326.285819,12.0,7.5,13.582400,1999,1,7,3,11,stepped around crates and truck tray fracture ...
2,WC5481426,1996-03-25 00:00:00+00:00,1996-04-14 00:00:00+00:00,30,1,2,0,0,709.10,0,38.0,5,CUT ON SHARP EDGE CUT LEFT THUMB,1700,2293.949087,20.0,7.6,18.660526,1996,3,25,0,0,cut on sharp edge cut left thumb


In [20]:
test.head(3)

,ClaimNumber,DateTimeOfAccident,DateReported,Age,Gender,MaritalStatus,DependentChildren,DependentsOther,WeeklyWages,PartTimeFullTime,HoursWorkedPerWeek,DaysWorkedPerWeek,ClaimDescription,InitialIncurredCalimsCost,ReportingDelay,HoursPerDay,HourlyWage,AccidentYear,AccidentMonth,AccidentDay,AccidentWeekday,AccidentHour,ClaimDescription_clean
0,WC8145235,2002-04-02 10:00:00+00:00,2002-05-07 00:00:00+00:00,26,1,1,1,0,600.18,0,40.0,5,CAUGHT RIGHT HAND WITH HAMMER BURN TO RIGHT HAND,5300,34.0,8.0,15.004500,2002,4,2,1,10,caught right hand with hammer burn to right hand
1,WC2005111,1988-04-06 16:00:00+00:00,1988-04-15 00:00:00+00:00,31,1,0,0,0,311.54,0,35.0,5,SPRAINED RIGHT ANKLE FRACTURE RIGHT ELBOW,2000,8.0,7.0,8.901143,1988,4,6,2,16,sprained right ankle fracture right elbow
2,WC6899143,1999-03-08 09:00:00+00:00,1999-04-04 00:00:00+00:00,57,1,0,0,0,1000.00,0,38.0,5,STRUCK HAMMER CRUSH INJURY FINGERS HAND,20000,26.0,7.6,26.315789,1999,3,8,0,9,struck hammer crush injury fingers hand


In [ ]:
sparse.save_npz(
    "../data/X_train.npz",
    X_train
)

sparse.save_npz(
    "../data/X_test.npz",
    X_test
)

y_train.to_csv(
    "../data/y_train.csv",
    index=False
)



Point à améliorer: pos-taging, embeddings, modele type bert